# Large-scale Pfam protein-collection and graph exploration

This notebook moves beyond the 12-protein PF00042 smoke test in two stages. Part A uses the original **205-domain controlled panel** containing six purposively chosen Pfam families in three clans. Part B is the main large-scale experiment: a reproducibly selected **960-domain master pool** containing 24 families in eight clans, from which balanced collections are repeatedly sampled.

Algorithm choice is only one experimental factor. DEDAL is disabled by default because it is not needed to investigate the more fundamental input and graph-construction questions.

## Questions examined

1. Does graph structure change as the protein collection grows?
2. Does mixing families within a clan differ from mixing unrelated clans?
3. How sensitive are isolates, components, density, and biological grouping to the graph rule?
4. Do conclusions persist across `top_k` and percentile thresholds?
5. Once those effects are visible, how much additional variation comes from Biopython versus BLAST?
6. Across repeated samples, what distribution of family- and clan-recovery NMI does each graph-construction rule produce?

In [ ]:
import json
import subprocess
import sys
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display

from protein_alignment_networks import (
    build_similarity_graph,
    graph_summary,
    read_fasta,
    save_graph_bundle,
)
from protein_alignment_networks.visualization import (
    draw_similarity_graph,
    shared_graph_layout,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

## 1. Visible experiment configuration

The default collection contains every eligible curated seed sequence when a family has fewer than 40, otherwise 40 diversity-selected representatives. Existing downloads and scores are reused.

In [ ]:
RUN_DATA_PREPARATION = True
RUN_SCORING = True
METHODS_TO_RUN = ('biopython', 'blast')  # add 'dedal' only intentionally
BLAST_THREADS = 4

BALANCED_COLLECTION_SIZES = (60, 120, 180)
GRAPH_RULES = {
    'top_k': (1, 2, 5, 10),
    'percentile': (0.90, 0.95, 0.98, 0.99),
}
VISUAL_RULE = 'top_k'
VISUAL_THRESHOLD = 5
RANDOM_SEED = 42

DATASET_DIRECTORY = PROJECT_ROOT / 'data/processed/pfam_clan_panel'
FASTA_PATH = DATASET_DIRECTORY / 'pfam_clan_panel.fasta'
METADATA_PATH = DATASET_DIRECTORY / 'pfam_clan_panel_metadata.tsv'
DATASET_MANIFEST_PATH = DATASET_DIRECTORY / 'pfam_clan_panel_manifest.json'
SCORE_DIRECTORY = PROJECT_ROOT / 'outputs/tables/pfam_clan_panel'
FIGURE_DIRECTORY = PROJECT_ROOT / 'outputs/figures/pfam_clan_panel'
GRAPH_DIRECTORY = PROJECT_ROOT / 'outputs/graphs/pfam_clan_panel'
FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
GRAPH_DIRECTORY.mkdir(parents=True, exist_ok=True)

## 2. Prepare the master protein collection

### What is being selected?

Each node is one **Pfam domain instance**, not necessarily one complete protein. A domain is a sequence region treated as a distinct structural and/or functional evolutionary unit. A Pfam family groups homologous domain instances using a curated seed alignment and a profile hidden Markov model (HMM); a Pfam clan groups more distantly related Pfam families using evidence such as sequence, structure, function, and profile-HMM similarity. Domains, families, and broader homologous superfamilies are standard biological concepts, but the exact family boundaries, names, HMMs, and clan assignments used here are Pfam's curated classification.

### How were the six families chosen?

The **family choice was purposive rather than random**. Two families were manually chosen from each of three Pfam clans—globin, protein kinase, and SH3—to create a controlled hierarchy with three known relationship classes: same family, different family within the same clan, and different clan. The exact accessions are hard-coded in `DEFAULT_FAMILIES` in `scripts/prepare_pfam_clan_panel.py`; no algorithm searched all of Pfam and selected these six families. To reproduce this exact panel manually, use PF00042 and PF01152 from CL0090, PF00069 and PF07714 from CL0016, and PF00018 and PF08239 from CL0010. This is an exploration panel, not a representative sample of Pfam or of the protein universe, so conclusions from it must later be checked on additional independently chosen collections.

### How were the individual domain instances chosen?

Within each chosen family, selection is systematic, deterministic, and reproducible:

1. Download the official curated Pfam **seed alignment** through InterPro.
2. Extract the annotated domain coordinates and discard empty sequences, sequences containing non-standard amino-acid symbols, coordinate/length mismatches, and duplicate ungapped domain sequences.
3. If at most 40 eligible unique seed domains remain, retain all of them.
4. Otherwise, select 40 diverse representatives by farthest-point sampling on the Pfam seed alignment. Calculate distance as `1 - identity` over alignment columns where both candidates have amino acids. Begin with the domain closest to the family's median ungapped length; if tied, take the lexicographically smallest node ID. For every remaining candidate, calculate its distance to the nearest selected domain, select the candidate for which that nearest-selected distance is greatest, and repeat until 40 have been selected. The implemented tie-break at this stage is the lexicographically greatest node ID.
5. Remove alignment gaps before scoring. Biopython and BLAST therefore receive ordinary unaligned domain sequences; the Pfam alignment is used only for representative selection.

This procedure deliberately favours **sequence diversity**, not taxonomic prevalence or a random population sample. The resulting counts are 40 PF00042 globins, 8 PF01152 bacterial-like globins, 37 PF00069 kinase domains, 40 PF07714 tyrosine/serine-threonine kinase domains, 40 PF00018 SH3 domains, and 40 PF08239 bacterial SH3 domains: 205 domain instances in total.

In [ ]:
if RUN_DATA_PREPARATION:
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / 'scripts/prepare_pfam_clan_panel.py')],
        cwd=PROJECT_ROOT,
        check=True,
    )

metadata = pd.read_csv(METADATA_PATH, sep='\t')
dataset_manifest = json.loads(DATASET_MANIFEST_PATH.read_text())
sequences = read_fasta(FASTA_PATH)
assert set(sequences) == set(metadata['node_id'])
print(
    f"{len(metadata)} protein domains; "
    f"{dataset_manifest['unique_pair_count']:,} unique pairs; "
    f"{metadata['family_accession'].nunique()} families; "
    f"{metadata['clan_accession'].nunique()} clans"
)
display(
    metadata.groupby(
        ['clan_accession', 'clan_name', 'family_accession', 'family_name'],
        dropna=False,
    )
    .agg(proteins=('node_id', 'size'), median_length=('domain_length', 'median'))
    .reset_index()
)

## 3. Produce or reuse pairwise scores

Biopython and BLAST score all pairs in seconds at this scale. DEDAL remains available in the pipeline but is deliberately omitted from this exploration until its cost is justified by a specific question.

In [ ]:
if RUN_SCORING:
    subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / 'scripts/run_pair_scores.py'),
            '--fasta', str(FASTA_PATH),
            '--dataset-manifest', str(DATASET_MANIFEST_PATH),
            '--output-directory', str(SCORE_DIRECTORY),
            '--experiment-name', 'Pfam clan panel exploration',
            '--methods', *METHODS_TO_RUN,
            '--threads', str(BLAST_THREADS),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

pair_scores = pd.read_csv(SCORE_DIRECTORY / 'paired_scores.tsv', sep='\t')
score_manifest = json.loads((SCORE_DIRECTORY / 'run_manifest.json').read_text())
METHOD_COLUMNS = {
    method: column
    for method, column in {
        'biopython': 'biopython_score',
        'blast': 'blast_bit_score',
        'dedal': 'dedal_sw_score',
    }.items()
    if column in pair_scores
}
print('Completed methods:', ', '.join(METHOD_COLUMNS))
print('Recorded runtimes (seconds):', score_manifest['runtimes_seconds'])
assert len(pair_scores) == dataset_manifest['unique_pair_count']

## 4. Add biological context to every protein pair

Pairs are labelled as belonging to the same Pfam family, different families within the same clan, or different clans. These labels are reference annotations for interpreting graph structure, not inputs to the alignment algorithms.

In [ ]:
node_metadata = metadata.set_index('node_id')
pair_analysis = pair_scores.copy()
for endpoint in ('a', 'b'):
    pair_analysis[f'family_{endpoint}'] = pair_analysis[f'protein_{endpoint}'].map(
        node_metadata['family_accession']
    )
    pair_analysis[f'clan_{endpoint}'] = pair_analysis[f'protein_{endpoint}'].map(
        node_metadata['clan_accession']
    )
    pair_analysis[f'length_{endpoint}'] = pair_analysis[f'protein_{endpoint}'].map(
        node_metadata['domain_length']
    )
pair_analysis['relationship'] = np.select(
    [
        pair_analysis['family_a'] == pair_analysis['family_b'],
        pair_analysis['clan_a'] == pair_analysis['clan_b'],
    ],
    ['same family', 'same clan, different family'],
    default='different clan',
)
RELATIONSHIP_ORDER = [
    'same family', 'same clan, different family', 'different clan'
]
display(
    pair_analysis.groupby('relationship', observed=True)
    .agg(pairs=('protein_a', 'size'), blast_reported=('blast_bit_score', 'count'))
    .reindex(RELATIONSHIP_ORDER)
)

## 5. Explore scores, coverage, and biological separation

Raw values remain on their native method scales. The useful question here is whether score distributions separate the three biological relationship classes and where BLAST declines to report a hit.

In [ ]:
figure, axes = plt.subplots(1, len(METHOD_COLUMNS), figsize=(7 * len(METHOD_COLUMNS), 5))
axes = np.atleast_1d(axes)
for axis, (method, column) in zip(axes, METHOD_COLUMNS.items(), strict=True):
    values = [
        pair_analysis.loc[pair_analysis['relationship'] == label, column].dropna()
        for label in RELATIONSHIP_ORDER
    ]
    axis.boxplot(values, tick_labels=RELATIONSHIP_ORDER, showfliers=False)
    axis.set_title(f'{method}: native score by relationship')
    axis.set_ylabel(column.replace('_', ' '))
    axis.tick_params(axis='x', rotation=18)
figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / 'scores_by_relationship.png', dpi=180)
plt.show()

coverage_rows = []
for relationship in RELATIONSHIP_ORDER:
    group = pair_analysis.loc[pair_analysis['relationship'] == relationship]
    for method, column in METHOD_COLUMNS.items():
        coverage_rows.append({
            'relationship': relationship,
            'method': method,
            'pairs': len(group),
            'reported_pairs': int(group[column].notna().sum()),
            'coverage_fraction': group[column].notna().mean(),
        })
coverage = pd.DataFrame(coverage_rows)
display(coverage)

## 6. Treat method agreement as one diagnostic

Spearman correlation is calculated within each biological relationship class as well as overall. This avoids allowing the easy same-family versus different-clan separation to create a misleadingly high global correlation.

In [ ]:
correlation_rows = []
scopes = [('all pairs', pair_analysis), *pair_analysis.groupby('relationship')]
for scope, group in scopes:
    for (method_a, column_a), (method_b, column_b) in combinations(
        METHOD_COLUMNS.items(), 2
    ):
        complete = group[[column_a, column_b]].dropna()
        correlation_rows.append({
            'scope': scope,
            'method_a': method_a,
            'method_b': method_b,
            'spearman_rho': complete[column_a].rank(method='average').corr(
                complete[column_b].rank(method='average')
            ),
            'comparable_pairs': len(complete),
        })
correlations = pd.DataFrame(correlation_rows)
display(correlations)

## 7. Define many protein collections from one scored master pool

Scoring the master pool once lets us change the input collection without recomputing alignments. The scenarios below include size-controlled diversity samples, individual clans, and individual Pfam families.

In [ ]:
def diversity_ranked_ids(table, size):
    if not 1 <= size <= len(table):
        raise ValueError('collection size is outside the available range')
    ordered = table.sort_values(
        ['family_selection_rank', 'family_accession', 'node_id']
    )
    return ordered.head(size)['node_id'].tolist()


collections = {f'all_{len(metadata)}': metadata['node_id'].tolist()}
for size in BALANCED_COLLECTION_SIZES:
    if size < len(metadata):
        collections[f'diversity_{size}'] = diversity_ranked_ids(metadata, size)
for clan, group in metadata.groupby('clan_accession'):
    collections[f'clan_{clan}'] = group['node_id'].tolist()
for family, group in metadata.groupby('family_accession'):
    collections[f'family_{family}'] = group['node_id'].tolist()

collection_rows = []
for collection_id, identifiers in collections.items():
    subset = metadata.loc[metadata['node_id'].isin(identifiers)]
    collection_rows.append({
        'collection_id': collection_id,
        'proteins': len(subset),
        'possible_pairs': len(subset) * (len(subset) - 1) // 2,
        'families': subset['family_accession'].nunique(),
        'clans': subset['clan_accession'].nunique(),
    })
collection_definitions = pd.DataFrame(collection_rows).sort_values(
    ['proteins', 'collection_id'], ascending=[False, True]
)
collection_definitions.to_csv(
    SCORE_DIRECTORY / 'collection_definitions.tsv', sep='\t', index=False
)
display(collection_definitions)

## 8. Sweep graph-construction choices

`top_n` means keeping exactly the strongest `n` pairs globally; the old value 13 was only a tiny smoke test and does not scale with collection size. This notebook instead emphasises `top_k`, the union of every protein's strongest `k` available neighbours, plus percentile thresholds that retain the upper tail of each method's available scores.

In [ ]:
def edge_label_fraction(graph, attribute):
    if graph.number_of_edges() == 0:
        return np.nan
    matches = sum(
        graph.nodes[a].get(attribute) == graph.nodes[b].get(attribute)
        for a, b in graph.edges
    )
    return matches / graph.number_of_edges()


graph_cache = {}
summary_rows = []
for collection_id, identifiers in collections.items():
    identifier_set = set(identifiers)
    subset_pairs = pair_scores.loc[
        pair_scores['protein_a'].isin(identifier_set)
        & pair_scores['protein_b'].isin(identifier_set)
    ]
    subset_metadata = metadata.loc[metadata['node_id'].isin(identifier_set)]
    for method, score_column in METHOD_COLUMNS.items():
        for rule, thresholds in GRAPH_RULES.items():
            for threshold in thresholds:
                graph = build_similarity_graph(
                    subset_pairs, identifiers,
                    method=method, score_column=score_column,
                    threshold=threshold, threshold_rule=rule,
                    node_metadata=subset_metadata, node_id_column='node_id',
                )
                key = (collection_id, method, rule, threshold)
                graph_cache[key] = graph
                summary = graph_summary(graph)
                component_sizes = sorted(
                    (len(component) for component in nx.connected_components(graph)),
                    reverse=True,
                )
                summary_rows.append({
                    **summary,
                    'collection_id': collection_id,
                    'score_column': score_column,
                    'largest_component': component_sizes[0],
                    'same_family_edge_fraction': edge_label_fraction(
                        graph, 'family_accession'
                    ),
                    'same_clan_edge_fraction': edge_label_fraction(
                        graph, 'clan_accession'
                    ),
                })

graph_summaries = pd.DataFrame(summary_rows)
graph_summaries.to_csv(
    SCORE_DIRECTORY / 'scenario_graph_summaries.tsv', sep='\t', index=False
)
display(graph_summaries.head())
print(f'Built {len(graph_cache)} scenario graphs in memory')

## 9. Examine size and threshold sensitivity

A graph conclusion is credible only if it is not an accident of one threshold. These plots expose how graph density, isolates, and biological edge composition move across the configured rule sweep.

In [ ]:
master_collection = f'all_{len(metadata)}'
figure, axes = plt.subplots(2, 2, figsize=(13, 9))
metrics = [
    ('density', 'Graph density'),
    ('isolates', 'Isolated proteins'),
    ('same_family_edge_fraction', 'Edges within same family'),
    ('same_clan_edge_fraction', 'Edges within same clan'),
]
top_k_summary = graph_summaries.loc[
    (graph_summaries['collection_id'] == master_collection)
    & (graph_summaries['threshold_rule'] == 'top_k')
]
for axis, (metric, label) in zip(axes.flat, metrics, strict=True):
    for method, group in top_k_summary.groupby('method'):
        axis.plot(group['threshold'], group[metric], marker='o', label=method)
    axis.set_xlabel('k strongest neighbours nominated per protein')
    axis.set_ylabel(label)
    axis.legend(frameon=False)
figure.suptitle(f'Top-k sensitivity on {master_collection}')
figure.tight_layout()
figure.savefig(FIGURE_DIRECTORY / 'top_k_sensitivity.png', dpi=180)
plt.show()

## 10. Visualize comparable graphs

Both method graphs use the same node positions, calculated from their union, and the same Pfam-family colours. Without a shared layout, visual differences could be caused by the drawing algorithm rather than the graph itself.

In [ ]:
visual_graphs = {
    method: graph_cache[(master_collection, method, VISUAL_RULE, VISUAL_THRESHOLD)]
    for method in METHOD_COLUMNS
}
positions = shared_graph_layout(visual_graphs.values(), seed=RANDOM_SEED)
families = sorted(metadata['family_accession'].unique())
palette = plt.get_cmap('tab20', len(families))
family_colors = {family: palette(index) for index, family in enumerate(families)}
figure, axes = plt.subplots(1, len(visual_graphs), figsize=(9 * len(visual_graphs), 8))
axes = np.atleast_1d(axes)
for axis, (method, graph) in zip(axes, visual_graphs.items(), strict=True):
    draw_similarity_graph(
        graph, positions=positions, color_attribute='family_accession',
        category_colors=family_colors,
        title=f'{method}: {VISUAL_RULE}={VISUAL_THRESHOLD}',
        ax=axis, node_size=48, edge_alpha=0.16,
    )
figure.suptitle(f'{master_collection}: shared-layout protein graphs')
figure.tight_layout()
figure.savefig(
    FIGURE_DIRECTORY / f'{master_collection}_{VISUAL_RULE}_{VISUAL_THRESHOLD}.png',
    dpi=200,
)
plt.show()

for method, graph in visual_graphs.items():
    save_graph_bundle(
        graph,
        GRAPH_DIRECTORY / f'{master_collection}_{method}_{VISUAL_RULE}_{VISUAL_THRESHOLD}',
    )

## 11. Quantify method edge overlap

Visual similarity is not a statistic. Jaccard overlap records the fraction of edges shared by each pair of available methods for every collection and graph rule, while leaving collection and threshold effects visible.

In [ ]:
def canonical_edges(graph):
    return {tuple(sorted(edge)) for edge in graph.edges}


overlap_rows = []
for collection_id in collections:
    for rule, thresholds in GRAPH_RULES.items():
        for threshold in thresholds:
            for method_a, method_b in combinations(METHOD_COLUMNS, 2):
                edges_a = canonical_edges(
                    graph_cache[(collection_id, method_a, rule, threshold)]
                )
                edges_b = canonical_edges(
                    graph_cache[(collection_id, method_b, rule, threshold)]
                )
                union = edges_a | edges_b
                overlap_rows.append({
                    'collection_id': collection_id,
                    'threshold_rule': rule,
                    'threshold': threshold,
                    'method_a': method_a, 'method_b': method_b,
                    'edges_a': len(edges_a), 'edges_b': len(edges_b),
                    'shared_edges': len(edges_a & edges_b),
                    'edge_jaccard': len(edges_a & edges_b) / len(union)
                    if union else np.nan,
                })
method_edge_overlap = pd.DataFrame(overlap_rows)
method_edge_overlap.to_csv(
    SCORE_DIRECTORY / 'method_edge_overlap.tsv', sep='\t', index=False
)
display(method_edge_overlap.loc[
    (method_edge_overlap['collection_id'] == master_collection)
    & (method_edge_overlap['threshold_rule'] == VISUAL_RULE)
])

## 12. Repeated sampling from a reproducibly selected Pfam panel

This is the main large-scale experiment proposed at the 28 August meeting. The sampling unit is a **Pfam seed domain instance**, not a whole protein. `scripts/select_pfam_family_panel.py` selected eight clans and three eligible families per clan from the official Pfam 38.2 catalogue using a fixed seed and explicit rejection rules. `scripts/prepare_pfam_clan_panel.py` then selected 40 valid, diverse domain instances per family, producing 960 nodes and 460,320 master pairs.

From this pool, `scripts/generate_pfam_collections.py` created 900 balanced collections: 2, 4, or 8 clans; 2 or 3 families per clan; 10, 20, or 40 domains per family; and 50 replicates per condition. Family and clan labels determine the balanced sampling design, but they are **not** supplied to graph construction or Louvain community detection. They are revealed only afterwards to calculate NMI and the chance-adjusted AMI. Every family selection, rejected candidate, domain membership, random seed, input checksum, and graph setting is saved outside the notebook.

In [ ]:
LARGE_PANEL_DIRECTORY = PROJECT_ROOT / 'data/processed/pfam_large_panel'
RESAMPLING_DIRECTORY = LARGE_PANEL_DIRECTORY / 'resampling'
RECOVERY_DIRECTORY = PROJECT_ROOT / 'outputs/tables/pfam_large_panel_graph_recovery_initial'
LARGE_FIGURE_DIRECTORY = PROJECT_ROOT / 'outputs/figures/pfam_large_panel'
LARGE_FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)

large_metadata = pd.read_csv(
    LARGE_PANEL_DIRECTORY / 'pfam_large_panel_metadata.tsv', sep='\t'
)
sampling_manifest = pd.read_csv(
    RESAMPLING_DIRECTORY / 'collection_manifest.tsv', sep='\t'
)
recovery_results = pd.read_csv(
    RECOVERY_DIRECTORY / 'graph_recovery_results.tsv', sep='\t'
)

print(
    f'{len(large_metadata)} domains; '
    f"{large_metadata['family_accession'].nunique()} families; "
    f"{large_metadata['clan_accession'].nunique()} clans; "
    f'{len(sampling_manifest)} predeclared sampled collections'
)
display(
    large_metadata.groupby(
        ['clan_accession', 'clan_name', 'family_accession', 'family_name'],
        as_index=False,
    ).agg(domains=('node_id', 'size'))
)

### First NMI distribution

The first completed slice fixes the input at four clans, two families per clan, and 20 domains per family: 160 nodes per graph and 50 independently sampled collections. It uses the Biopython score only and varies `top_k` over 1, 2, 5, and 10. Holding the input design and scoring method fixed means differences between these curves can be attributed to `top_k` rather than to differently sampled proteins. This is an initial diagnostic slice, not yet the final comparison across all 900 collections.

In [ ]:
nmi_summary = (
    recovery_results
    .groupby(['reference_label', 'threshold'], as_index=False)
    .agg(
        graphs=('graph_id', 'nunique'),
        median_nmi=('nmi', 'median'),
        lower_quartile_nmi=('nmi', lambda values: values.quantile(0.25)),
        upper_quartile_nmi=('nmi', lambda values: values.quantile(0.75)),
        median_ami=('ami', 'median'),
        median_components=('connected_components', 'median'),
        median_communities=('inferred_community_count', 'median'),
    )
)
display(nmi_summary)

bins = np.linspace(0, 1, 31)
figure, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True)
for axis, reference_label, title in zip(
    axes,
    ('family_accession', 'clan_accession'),
    ('Recovery of Pfam families', 'Recovery of Pfam clans'),
):
    subset = recovery_results.loc[
        recovery_results['reference_label'] == reference_label
    ]
    for threshold, group in subset.groupby('threshold'):
        axis.hist(
            group['nmi'], bins=bins, density=True, histtype='step',
            linewidth=2, label=f'top_k={int(threshold)}',
        )
    axis.set_title(title)
    axis.set_xlabel('Normalized mutual information (NMI)')
    axis.set_ylabel('Empirical density')
    axis.set_xlim(0, 1)
    axis.legend()
figure.suptitle('Repeated 160-domain samples: graph frequency by NMI')
figure.tight_layout()
figure.savefig(
    LARGE_FIGURE_DIRECTORY / 'initial_top_k_nmi_distributions.png', dpi=200
)
plt.show()

## 13. The complete predeclared design

Section 12 reported one fixed input condition. This section loads the **complete** predeclared experiment, run on 31 August 2026.

Two runs were executed by `scripts/run_resampled_graph_experiment.py`, both over the same 900 sampled collections so that every comparison is paired on identical protein sets:

| Run | Rules | Graphs | Evaluation rows |
|---|---|---:|---:|
| `..._graph_recovery_full` | `top_k` 1, 2, 5, 10 and `percentile` 0.90, 0.95, 0.98, 0.99 | 14,400 | 28,800 |
| `..._graph_recovery_matched_density` | `target_density` 0.005, 0.01, 0.02, 0.05 | 7,200 | 14,400 |

Both methods, both reference labels, and all 18 composition cells are present in every combination, so the design is balanced. `scripts/summarize_resampled_graph_experiment.py` then derived the summary tables and figures loaded below; every input and output is recorded with a SHA256 checksum in the run manifests.

The long runs are checkpointed every 25 collections and support `--resume`, so an interrupted run does not lose completed work.

In [ ]:
FULL_RECOVERY_DIRECTORY = (
    PROJECT_ROOT / 'outputs/tables/pfam_large_panel_graph_recovery_full'
)
MATCHED_RECOVERY_DIRECTORY = (
    PROJECT_ROOT / 'outputs/tables/pfam_large_panel_graph_recovery_matched_density'
)
SUMMARY_DIRECTORY = PROJECT_ROOT / 'outputs/tables/pfam_large_panel_graph_summary'
FULL_FIGURE_DIRECTORY = PROJECT_ROOT / 'outputs/figures/pfam_large_panel_full'

full_results = pd.read_csv(
    FULL_RECOVERY_DIRECTORY / 'graph_recovery_results.tsv', sep='\t'
)
matched_results = pd.read_csv(
    MATCHED_RECOVERY_DIRECTORY / 'graph_recovery_results.tsv', sep='\t'
)
all_results = pd.concat([full_results, matched_results], ignore_index=True)

rule_summary = pd.read_csv(SUMMARY_DIRECTORY / 'rule_summary.tsv', sep='\t')
factor_summary = pd.read_csv(SUMMARY_DIRECTORY / 'factor_summary.tsv', sep='\t')
paired_method_summary = pd.read_csv(
    SUMMARY_DIRECTORY / 'paired_method_summary.tsv', sep='\t'
)
collection_uniqueness = pd.read_csv(
    SUMMARY_DIRECTORY / 'collection_uniqueness.tsv', sep='\t'
)

for name, table in [('full', full_results), ('matched-density', matched_results)]:
    print(
        f'{name:16s} graphs={table["graph_id"].nunique():6d}  '
        f'rows={len(table):6d}  collections={table["collection_id"].nunique():4d}  '
        f'missing NMI={int(table["nmi"].isna().sum())}  '
        f'zero-edge graphs={int((table["edges"] == 0).sum())}'
    )

display(
    all_results
    .groupby(['threshold_rule', 'method'], as_index=False)
    .agg(graphs=('graph_id', 'nunique'), rows=('nmi', 'size'))
)

## 14. Why equal score percentiles do not compare two methods

The first result from the full run looks like a large method effect: at the nominal 90th score percentile, Biopython reaches a median family NMI of about **0.908** while BLAST reaches about **0.662**.

That comparison is not fair, and the reason is visible in the realised graphs. A percentile is taken over the scores a method actually produced. BLAST returns no hit for many distant pairs, so its available-score pool is much smaller, and "the top 10% of available hits" is a far sparser graph than Biopython's: median density **0.0085** against **0.0999**, or 164 edges against 1,998 on the same nodes.

The two graphs being compared therefore differ by roughly a factor of twelve in edge count. Denser graphs recover labels better here, so most of the apparent method gap is a density difference wearing a method's name. This is a property of the *rule*, not evidence about the algorithms.

In [ ]:
percentile_view = rule_summary.loc[
    (rule_summary['threshold_rule'] == 'percentile')
    & (rule_summary['reference_label'] == 'family_accession')
]
display(
    percentile_view[
        ['method', 'threshold', 'median_density', 'median_edges', 'median_nmi']
    ].sort_values(['threshold', 'method'], ignore_index=True)
)

figure, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for axis, column, title, ylabel in zip(
    axes,
    ('median_density', 'median_nmi'),
    ('Realised graph density', 'Family recovery'),
    ('Median realised density', 'Median NMI'),
):
    for method, group in percentile_view.groupby('method'):
        ordered = group.sort_values('threshold')
        axis.plot(
            (1 - ordered['threshold']) * 100, ordered[column],
            marker='o', label=method,
        )
    axis.set_title(title)
    axis.set_xlabel('Top fraction of available scores retained (%)')
    axis.set_ylabel(ylabel)
    axis.legend()
figure.suptitle('Equal percentiles produce unequal graphs')
figure.tight_layout()
plt.show()

print(
    'Full three-panel version saved by the summary script at '
    f'{FULL_FIGURE_DIRECTORY / "percentile_density_mismatch.png"}'
)

## 15. Matching the edge budget across methods

To separate the two effects, `build_similarity_graph` gained a `target_density` rule: retain exactly the strongest `ceil(density x possible_pairs)` available pairs, so both methods are given the **same edge budget on the same nodes**. The rule also records `selection_shortfall`, the number of edges a method could not supply.

Once the budget is equal, the method difference nearly disappears. Median absolute paired NMI difference, family recovery:

| Rule | Median absolute difference |
|---|---:|
| `percentile` 0.90 | 0.242 |
| `target_density` 0.005 | 0.0012 |
| `target_density` 0.01 | 0.0018 |
| `target_density` 0.02 | 0.0038 |
| `target_density` 0.05 | 0.0081 |

That is a roughly two-hundred-fold reduction. The large percentile gap was almost entirely unmatched sparsity.

**One caveat must be carried with this result.** The matching is not complete everywhere. At the 5% budget, BLAST cannot supply enough scored pairs in the largest and most diverse collections: 320 of the 14,400 matched-density rows carry a non-zero shortfall, all of them BLAST at `target_density=0.05`, concentrated in the eight-clan designs. In the 8-clan x 3-family x 40-domain cell the median shortfall is 3,640 edges against a 23,016-edge target, about 16% short. Those graphs are *not* density-matched, so the 5% row above averages over cells where the matching silently failed. The 0.5%, 1% and 2% budgets have no shortfall anywhere and carry the conclusion on their own.

In [ ]:
display(
    paired_method_summary.loc[
        paired_method_summary['reference_label'] == 'family_accession',
        ['threshold_rule', 'threshold', 'pairs',
         'median_absolute_nmi_difference', 'median_density_difference'],
    ].sort_values(['threshold_rule', 'threshold'], ignore_index=True)
)

shortfall = matched_results.loc[matched_results['selection_shortfall'] > 0]
print(
    f'{len(shortfall)} of {len(matched_results)} matched-density rows have a '
    f'non-zero edge-budget shortfall; methods affected: '
    f'{sorted(shortfall["method"].unique())}; '
    f'densities affected: {sorted(shortfall["threshold"].unique())}'
)
display(
    shortfall
    .groupby(
        ['method', 'threshold', 'clans_per_collection',
         'families_per_clan', 'domains_per_family'],
        as_index=False,
    )
    .agg(
        rows=('nmi', 'size'),
        median_target_edges=('target_edges', 'median'),
        median_actual_edges=('edges', 'median'),
        median_shortfall=('selection_shortfall', 'median'),
    )
)

# Recompute the paired difference using only graphs where matching truly held.
paired = matched_results.pivot_table(
    index=['collection_id', 'threshold', 'reference_label'],
    columns='method',
    values='nmi',
).reset_index()
affected = (
    matched_results.loc[
        matched_results['selection_shortfall'] > 0,
        ['collection_id', 'threshold'],
    ]
    .drop_duplicates()
)
affected_keys = set(map(tuple, affected.to_numpy()))
paired['has_shortfall'] = [
    (collection, threshold) in affected_keys
    for collection, threshold in zip(paired['collection_id'], paired['threshold'])
]
paired['absolute_difference'] = (paired['blast'] - paired['biopython']).abs()

shortfall_effect = []
for (reference_label, threshold), group in paired.groupby(
    ['reference_label', 'threshold']
):
    fully_matched = group.loc[~group['has_shortfall']]
    shortfall_effect.append({
        'reference_label': reference_label,
        'threshold': threshold,
        'all_pairs': len(group),
        'median_absolute_difference_all': group['absolute_difference'].median(),
        'fully_matched_pairs': len(fully_matched),
        'median_absolute_difference_matched_only':
            fully_matched['absolute_difference'].median(),
    })
display(pd.DataFrame(shortfall_effect))

## 16. Are the repeated collections actually independent?

The design declares 50 replicates per composition cell, but a replicate is only informative if it draws a different set of domains. The master pool holds 40 domain instances per family, so the largest cells ask for every domain the pool contains.

The uniqueness audit shows where this bites. In the **8 clans x 3 families x 40 domains** cell, all 50 "replicates" resolve to a *single* node set: that design exhausts the 960-domain pool, so those 50 graphs differ only through the Louvain random seed, not through sampling. The 2x3x40 and 4x3x40 cells are partially degenerate (23 and 30 unique sets of 50), and 2x2x40 loses four.

The remaining 14 cells are fully unique. Any distributional claim, and any variance estimate, must exclude or explicitly caveat the four affected cells: they measure algorithmic jitter, not sampling variability.

In [ ]:
display(collection_uniqueness)

degenerate = collection_uniqueness.loc[
    collection_uniqueness['duplicate_collection_count'] > 0
]
print(
    f'{len(degenerate)} of {len(collection_uniqueness)} composition cells contain '
    f'duplicate node sets, affecting '
    f'{int(degenerate["duplicate_collection_count"].sum())} declared collections.'
)
print(
    'Fully independent cells: '
    f'{int((collection_uniqueness["duplicate_collection_count"] == 0).sum())}'
)

## 17. Which experimental factor matters most?

Question 5 of this notebook asked how much variation comes from the choice of alignment method once the other factors are visible. The full design can now answer it directly.

For family recovery, holding the other factors fixed and measuring the median-to-median spread within each factor:

| Factor varied | Median spread in NMI |
|---|---:|
| Graph-construction rule (8 rules) | **0.325** |
| Collection composition (18 cells) | **0.274** |
| Alignment method (Biopython vs BLAST) | 0.029 |
| — method, under `top_k` rules only | **0.014** |
| — method, under `percentile` rules only | 0.154 |

The graph rule and the composition of the protein collection each move recovery by an order of magnitude more than the choice between Biopython and BLAST. The apparent method effect is itself dominated by the percentile artefact of Section 14: restricted to `top_k`, where both methods build graphs of comparable size, the method spread falls to 0.014.

For this project's purposes the two exact alignment methods are close to interchangeable as *edge scores*; what changes the answer is how edges are selected and which proteins are in the collection.

In [ ]:
family_medians = (
    full_results.loc[full_results['reference_label'] == 'family_accession']
    .groupby(
        ['clans_per_collection', 'families_per_clan', 'domains_per_family',
         'method', 'threshold_rule', 'threshold'],
        as_index=False,
    )
    .agg(median_nmi=('nmi', 'median'))
)
composition_columns = [
    'clans_per_collection', 'families_per_clan', 'domains_per_family',
]
rule_columns = ['threshold_rule', 'threshold']


def spread_holding_fixed(table, held_fixed):
    """Median and largest range of median NMI within each held-fixed group.

    Everything not named in ``held_fixed`` is the factor being varied, so the
    range inside a group is that factor's effect at one fixed setting of the
    others.
    """
    ranges = table.groupby(held_fixed)['median_nmi'].agg(
        lambda values: values.max() - values.min()
    )
    return ranges.median(), ranges.max(), len(ranges)


effects = {
    'graph rule': spread_holding_fixed(
        family_medians, composition_columns + ['method']
    ),
    'collection composition': spread_holding_fixed(
        family_medians, ['method'] + rule_columns
    ),
    'alignment method': spread_holding_fixed(
        family_medians, composition_columns + rule_columns
    ),
}
for rule in ('top_k', 'percentile'):
    effects[f'alignment method | {rule} only'] = spread_holding_fixed(
        family_medians.loc[family_medians['threshold_rule'] == rule],
        composition_columns + rule_columns,
    )

display(
    pd.DataFrame(
        [
            {'factor varied': name, 'median spread': median_spread,
             'largest spread': largest_spread, 'comparisons': comparisons}
            for name, (median_spread, largest_spread, comparisons)
            in effects.items()
        ]
    )
)

## Interpretation: what the full experiment settled, and what it did not

**Answered by this notebook.**

1. *Can graphs recover Pfam labels from an unlabelled collection?* Yes, and substantially. Under `top_k=10`, median family NMI reaches about 0.96 and median clan NMI about 0.76, with labels withheld from both graph construction and community detection. Family structure is recovered markedly better than the broader clan structure, which is the expected ordering.
2. *Are conclusions stable across thresholds?* The direction is stable — denser graphs recover families better across every rule tested — but the magnitude is not. NMI moves by roughly 0.33 across the eight rules, so no single `top_k` or percentile can yet be declared a default.
3. *Does the alignment method matter?* Much less than either the graph rule or the collection composition (0.014 versus 0.325 and 0.274 under `top_k`). Biopython and BLAST are near-interchangeable as edge scores at matched density.
4. *Does BLAST missingness matter?* Yes, but as a *density* effect on graph construction rather than as a recovery deficit. Any rule defined on the distribution of available scores silently compares graphs of different sizes across methods; rules defined on an edge budget (`top_k`, `target_density`) do not.

**Open, and needed before any default is chosen.**

1. The four partially or wholly degenerate composition cells (Section 16) must be excluded from distributional claims, or the master pool enlarged beyond 40 domains per family so the largest designs can draw genuine replicates.
2. The `target_density=0.05` comparison is incompletely matched for BLAST in the eight-clan cells (Section 15) and should be reported with that caveat or re-run at a budget BLAST can supply throughout.
3. Correlation-warning thresholds remain unset, and are still deliberately unset until they can be justified from the score distributions rather than chosen for convenience.
4. The collection types are still all Pfam seed domains. Complete proteins, taxonomically controlled samples, random controls, and synthetic sequences generated along a known tree remain outstanding, and the random control in particular is what would establish a floor for these NMI values.
5. Graph *structure* has not yet been compared across methods and rules beyond density, components, and community counts. Degree distributions, clustering, path structure, and the spectral and triangle statistics named in the proposal are still to do, and belong in Notebook 04.

The representative GraphML files can also be opened interactively in Gephi or Cytoscape, but the shared-layout figures above are the reproducible visual comparison used by this project.